# Patient 1 – Riemannian Geometry Motor Imagery Classification

**Pipeline:** Notch + Bandpass → Epoch → Covariance matrices → Riemannian classifier  
**Condition A:** Train on `P1_pre_training`, evaluate on `P1_pre_test`  
**Condition B:** Train on `P1_post_training`, evaluate on `P1_post_test`  
**Classifiers:** MDM (Minimum Distance to Mean) and Tangent Space + LDA  
**Labels:** 0 = Right Hand, 1 = Left Hand

```
pip install pyriemann
```

In [ ]:
import os
import scipy.io
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, iirnotch
import mne
from pyriemann.estimation import Covariances
from pyriemann.classification import MDM
from pyriemann.tangentspace import TangentSpace
from pyriemann.utils.mean import mean_covariance
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

mne.set_log_level('WARNING')

# ── CONFIG ────────────────────────────────────────────────────────────────────
DATA_DIR    = r"C:\Users\DIVI\Downloads\data"
FS          = 256     # Hz
TIME_AFTER  = 2.0     # seconds after cue to start epoch
EPOCH_DUR   = 4.0     # epoch length in seconds
N_CV_FOLDS  = 5
COV_EST     = 'oas'   # OAS-regularised covariance (robust for small n)

CHANNEL_NAMES = ['FC3','FCz','FC4','C5','C3','C1','Cz','C2','C4','C6',
                 'CP3','CP1','CPz','CP2','CP4','Pz']

info = mne.create_info(ch_names=CHANNEL_NAMES, sfreq=FS, ch_types='eeg')
info.set_montage(mne.channels.make_standard_montage('standard_1020'))
print('Setup complete.')

In [ ]:
# ── PREPROCESSING FUNCTIONS ───────────────────────────────────────────────────

def notch_filter(data, fs, freq, Q=30):
    b, a = iirnotch(freq, Q, fs)
    return filtfilt(b, a, data, axis=0)

def bandpass_filter(data, fs, lowcut=8.0, highcut=30.0, order=4):
    nyq = 0.5 * fs
    b, a = butter(order, [lowcut / nyq, highcut / nyq], btype='band')
    return filtfilt(b, a, data, axis=0)

def preprocess(y, fs=FS):
    y = notch_filter(y, fs, 50)
    y = notch_filter(y, fs, 60)
    y = bandpass_filter(y, fs, 8, 30)
    return y

def get_trigger_onsets(trig):
    t = trig.flatten()
    return np.where((t[1:] != 0) & (t[:-1] == 0))[0] + 1

def epoch_and_label(data, trig, fs=FS, time_after=TIME_AFTER, epoch_dur=EPOCH_DUR):
    """
    Returns
    -------
    X : (n_epochs, n_channels, n_times)  ← pyriemann expects this
    y : (n_epochs,)  0 = right, 1 = left
    """
    trig_flat   = trig.flatten()
    onset_samps = get_trigger_onsets(trig_flat)
    start_off   = int(time_after * fs)
    n_samps     = int(epoch_dur * fs)

    epochs, labels = [], []
    for idx in onset_samps:
        window = trig_flat[max(0, idx - 2): idx + 3]
        label  = int(np.sign(np.sum(window)))
        start  = idx + start_off
        end    = start + n_samps
        if end <= data.shape[0] and label != 0:
            epochs.append(data[start:end, :].T)       # (channels, time)
            labels.append(0 if label == -1 else 1)    # 0=right, 1=left

    return np.array(epochs), np.array(labels)

In [ ]:
# ── LOAD ALL FOUR P1 FILES ────────────────────────────────────────────────────

def load_mat(fname):
    mat  = scipy.io.loadmat(os.path.join(DATA_DIR, fname))
    return mat['y'].astype(float), mat['trig']

print('Loading …')
y_pre_tr,  trig_pre_tr  = load_mat('P1_pre_training.mat')
y_pre_te,  trig_pre_te  = load_mat('P1_pre_test.mat')
y_post_tr, trig_post_tr = load_mat('P1_post_training.mat')
y_post_te, trig_post_te = load_mat('P1_post_test.mat')

print('Filtering …')
y_pre_tr  = preprocess(y_pre_tr)
y_pre_te  = preprocess(y_pre_te)
y_post_tr = preprocess(y_post_tr)
y_post_te = preprocess(y_post_te)

print('Epoching …')
X_pre_tr,  L_pre_tr  = epoch_and_label(y_pre_tr,  trig_pre_tr)
X_pre_te,  L_pre_te  = epoch_and_label(y_pre_te,  trig_pre_te)
X_post_tr, L_post_tr = epoch_and_label(y_post_tr, trig_post_tr)
X_post_te, L_post_te = epoch_and_label(y_post_te, trig_post_te)

print()
print(f"{'Dataset':<18} {'Shape':>22}   right   left")
print('-' * 58)
for name, X, L in [
    ('pre_training',  X_pre_tr,  L_pre_tr),
    ('pre_test',      X_pre_te,  L_pre_te),
    ('post_training', X_post_tr, L_post_tr),
    ('post_test',     X_post_te, L_post_te),
]:
    print(f"{name:<18} {str(X.shape):>22}   {np.sum(L==0):>5}   {np.sum(L==1):>4}")

In [ ]:
# ── AUTOREJECT ────────────────────────────────────────────────────────────────
# Automatically detects and rejects/repairs bad epochs using cross-validation.
# Operates on MNE EpochsArray objects, then converts back to numpy for pyriemann.
#
# Install if needed:  pip install autoreject

from autoreject import AutoReject

def apply_autoreject(X, L, info, n_jobs=1):
    """
    X    : (n_epochs, n_channels, n_times)  numpy array
    L    : (n_epochs,)                      label array
    info : mne.Info object

    Returns cleaned X, L with rejected epochs removed.
    """
    epochs = mne.EpochsArray(X, info, tmin=0.0, verbose=False)
    ar = AutoReject(n_jobs=n_jobs, verbose=False)
    epochs_clean, reject_log = ar.fit_transform(epochs, return_log=True)

    kept_mask = ~reject_log.bad_epochs
    X_clean   = epochs_clean.get_data()
    L_clean   = L[kept_mask]

    n_rej = reject_log.bad_epochs.sum()
    print(f"  rejected {n_rej}/{len(L)} epochs ({n_rej/len(L)*100:.1f}%)")
    return X_clean, L_clean


print('Running AutoReject …')

print('  pre_training  ', end='')
X_pre_tr,  L_pre_tr  = apply_autoreject(X_pre_tr,  L_pre_tr,  info)

print('  pre_test      ', end='')
X_pre_te,  L_pre_te  = apply_autoreject(X_pre_te,  L_pre_te,  info)

print('  post_training ', end='')
X_post_tr, L_post_tr = apply_autoreject(X_post_tr, L_post_tr, info)

print('  post_test     ', end='')
X_post_te, L_post_te = apply_autoreject(X_post_te, L_post_te, info)

print('\nShapes after AutoReject:')
print(f"{'Dataset':<18} {'Shape':>22}   right   left")
print('-' * 58)
for name, X, L in [
    ('pre_training',  X_pre_tr,  L_pre_tr),
    ('pre_test',      X_pre_te,  L_pre_te),
    ('post_training', X_post_tr, L_post_tr),
    ('post_test',     X_post_te, L_post_te),
]:
    print(f"{name:<18} {str(X.shape):>22}   {np.sum(L==0):>5}   {np.sum(L==1):>4}")

In [ ]:
# ── RIEMANNIAN PIPELINES: FIT & EVALUATE ──────────────────────────────────────
#
# Covariances('oas')  – maps each epoch (ch × time) → SPD matrix (ch × ch)
#                       using Oracle Approximating Shrinkage regularisation
#
# MDM                 – classifies by Riemannian distance to per-class mean
# TangentSpace + LDA  – projects SPD matrices to the flat tangent space at
#                       the Riemannian mean, then applies LDA

def run_riemannian(X_train, L_train, X_test, L_test, condition=''):
    cv = StratifiedKFold(n_splits=N_CV_FOLDS, shuffle=True, random_state=42)
    results = {}

    for name, pipe in [
        ('MDM',      Pipeline([('cov', Covariances(COV_EST)),
                               ('clf', MDM(metric='riemann'))])),
        ('TS + LDA', Pipeline([('cov', Covariances(COV_EST)),
                               ('ts',  TangentSpace(metric='riemann')),
                               ('lda', LDA())])),
    ]:
        cv_scores = cross_val_score(pipe, X_train, L_train, cv=cv, scoring='accuracy')
        pipe.fit(X_train, L_train)
        acc_test  = accuracy_score(L_test, pipe.predict(X_test))

        print(f'  [{condition}]  {name:<12}  '
              f'CV: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}   '
              f'Test: {acc_test:.3f} ({acc_test*100:.1f}%)')

        results[name] = {
            'pipe': pipe,
            'cv_mean': cv_scores.mean(), 'cv_std': cv_scores.std(),
            'test_acc': acc_test,
        }

    chance = max(np.mean(L_test == 0), np.mean(L_test == 1))
    print(f'  [{condition}]  chance level: {chance:.3f}\n')
    return results


print('=== PRE CONDITION ===')
res_pre  = run_riemannian(X_pre_tr,  L_pre_tr,  X_pre_te,  L_pre_te,  'PRE')

print('=== POST CONDITION ===')
res_post = run_riemannian(X_post_tr, L_post_tr, X_post_te, L_post_te, 'POST')

In [ ]:
# ── CONFUSION MATRICES (TS + LDA) ─────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, res, X_te, L_te, cond in [
    (axes[0], res_pre,  X_pre_te,  L_pre_te,  'Pre'),
    (axes[1], res_post, X_post_te, L_post_te, 'Post'),
]:
    pipe = res['TS + LDA']['pipe']
    acc  = res['TS + LDA']['test_acc']
    cm   = confusion_matrix(L_te, pipe.predict(X_te))
    ConfusionMatrixDisplay(cm, display_labels=['Right', 'Left']).plot(
        ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'Patient 1 – {cond}  |  TS + LDA\nTest Acc: {acc*100:.1f}%', fontsize=11)

plt.suptitle('Riemannian Tangent Space + LDA', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('P1_riem_confusion.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: P1_riem_confusion.png')

In [ ]:
# ── CLASS-MEAN COVARIANCE HEATMAPS ────────────────────────────────────────────
#
# Shows the Riemannian mean covariance matrix per class (normalised to
# correlation scale). Strong off-diagonal values = correlated channels.
# Differences between right and left reflect lateralised mu/beta patterns.

def cov_to_corr(C):
    d = np.sqrt(np.diag(C))
    return C / np.outer(d, d)

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
cov_est = Covariances(COV_EST)

for row, (X_tr, L_tr, cond) in enumerate([
    (X_pre_tr,  L_pre_tr,  'Pre'),
    (X_post_tr, L_post_tr, 'Post'),
]):
    covs = cov_est.fit_transform(X_tr)         # (n_epochs, 16, 16)

    for col, (cls_idx, cls_name) in enumerate([(0, 'Right'), (1, 'Left')]):
        C_mean = mean_covariance(covs[L_tr == cls_idx], metric='riemann')
        C_corr = cov_to_corr(C_mean)

        ax = axes[row, col]
        im = ax.imshow(C_corr, cmap='RdBu_r', vmin=-1, vmax=1)
        ax.set_xticks(range(len(CHANNEL_NAMES)))
        ax.set_yticks(range(len(CHANNEL_NAMES)))
        ax.set_xticklabels(CHANNEL_NAMES, rotation=90, fontsize=7)
        ax.set_yticklabels(CHANNEL_NAMES, fontsize=7)
        ax.set_title(f'{cond} – {cls_name} Hand\n(Riemannian mean corr.)', fontsize=9)
        fig.colorbar(im, ax=ax, shrink=0.75)

    # Difference map: left minus right (in correlation space)
    for col_off, ref_cls, cmp_cls, ref_name, cmp_name in [
        (2, 0, 1, 'Right', 'Left'),
        (3, 1, 0, 'Left',  'Right'),
    ]:
        C_ref = cov_to_corr(mean_covariance(covs[L_tr == ref_cls], metric='riemann'))
        C_cmp = cov_to_corr(mean_covariance(covs[L_tr == cmp_cls], metric='riemann'))
        diff  = C_cmp - C_ref
        vabs  = np.abs(diff).max()

        ax = axes[row, col_off]
        im = ax.imshow(diff, cmap='RdBu_r', vmin=-vabs, vmax=vabs)
        ax.set_xticks(range(len(CHANNEL_NAMES)))
        ax.set_yticks(range(len(CHANNEL_NAMES)))
        ax.set_xticklabels(CHANNEL_NAMES, rotation=90, fontsize=7)
        ax.set_yticklabels(CHANNEL_NAMES, fontsize=7)
        ax.set_title(f'{cond} – {cmp_name} − {ref_name}\n(diff. map)', fontsize=9)
        fig.colorbar(im, ax=ax, shrink=0.75)

plt.suptitle('Class-Mean Covariance & Difference Maps (Pre / Post)', fontsize=13)
plt.tight_layout()
plt.savefig('P1_riem_covariance_maps.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: P1_riem_covariance_maps.png')

In [ ]:
# ── TANGENT SPACE 2D SCATTER ──────────────────────────────────────────────────
#
# Project all epochs (train + test) onto the 2D principal subspace of the
# tangent space. Well-separated clusters = good class discriminability.

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
colors = {0: '#4C72B0', 1: '#DD8452'}
markers = {'train': 'o', 'test': '^'}

for ax, X_tr, L_tr, X_te, L_te, cond in [
    (axes[0], X_pre_tr,  L_pre_tr,  X_pre_te,  L_pre_te,  'Pre'),
    (axes[1], X_post_tr, L_post_tr, X_post_te, L_post_te, 'Post'),
]:
    # Fit covariance estimator + tangent space on training data
    cov_est_viz = Covariances(COV_EST)
    ts_viz      = TangentSpace(metric='riemann')

    covs_tr = cov_est_viz.fit_transform(X_tr)
    T_tr    = ts_viz.fit_transform(covs_tr)

    covs_te = cov_est_viz.transform(X_te)
    T_te    = ts_viz.transform(covs_te)

    # PCA to 2D
    pca  = PCA(n_components=2)
    T_tr_2d = pca.fit_transform(T_tr)
    T_te_2d = pca.transform(T_te)

    for cls, cls_name in [(0, 'Right'), (1, 'Left')]:
        ax.scatter(T_tr_2d[L_tr == cls, 0], T_tr_2d[L_tr == cls, 1],
                   c=colors[cls], marker='o', s=60, alpha=0.7,
                   label=f'{cls_name} (train)')
        ax.scatter(T_te_2d[L_te == cls, 0], T_te_2d[L_te == cls, 1],
                   c=colors[cls], marker='^', s=80, alpha=0.9, edgecolors='k',
                   linewidths=0.5, label=f'{cls_name} (test)')

    var_exp = pca.explained_variance_ratio_
    ax.set_xlabel(f'PC1 ({var_exp[0]*100:.1f}% var)', fontsize=10)
    ax.set_ylabel(f'PC2 ({var_exp[1]*100:.1f}% var)', fontsize=10)
    ax.set_title(f'{cond}-Training: Tangent Space (2D PCA)', fontsize=11)
    ax.legend(fontsize=8, ncol=2)
    ax.spines[['top', 'right']].set_visible(False)

plt.suptitle('Riemannian Tangent Space Projection – Class Separability', fontsize=13)
plt.tight_layout()
plt.savefig('P1_riem_tangent_scatter.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: P1_riem_tangent_scatter.png')

In [ ]:
# ── ACCURACY COMPARISON ───────────────────────────────────────────────────────

methods     = ['MDM', 'TS + LDA']
conditions  = [('Pre', res_pre), ('Post', res_post)]
bar_colors  = {'MDM': '#55A868', 'TS + LDA': '#C44E52'}
x           = np.arange(len(conditions))
width       = 0.30

fig, ax = plt.subplots(figsize=(8, 5))

for i, method in enumerate(methods):
    test_accs = [res[method]['test_acc'] for _, res in conditions]
    cv_stds   = [res[method]['cv_std']   for _, res in conditions]
    offset    = (i - 0.5) * width

    bars = ax.bar(x + offset, test_accs, width,
                  label=method, color=bar_colors[method],
                  alpha=0.85, edgecolor='white')

    for bar, val in zip(bars, test_accs):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.02,
                f'{val*100:.1f}%',
                ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.axhline(0.5, color='gray', linestyle='--', alpha=0.6, label='Chance (50%)')
ax.set_xticks(x)
ax.set_xticklabels([c for c, _ in conditions], fontsize=12)
ax.set_ylim(0, 1.12)
ax.set_ylabel('Test Accuracy', fontsize=12)
ax.set_title('Patient 1 – Riemannian Geometry: Pre vs Post Rehabilitation', fontsize=13)
ax.legend(fontsize=10)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('P1_riem_accuracy.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: P1_riem_accuracy.png')

In [ ]:
# ── SUMMARY ───────────────────────────────────────────────────────────────────

print('=' * 60)
print('  PATIENT 1 – RIEMANNIAN GEOMETRY SUMMARY')
print('=' * 60)
print(f'  Epoch window  : {TIME_AFTER}s – {TIME_AFTER+EPOCH_DUR}s post-cue ({EPOCH_DUR}s)')
print(f'  Freq band     : 8–30 Hz (Mu + Beta)')
print(f'  Covariance    : {COV_EST.upper()} regularisation')
print()
print(f"  {'Condition':<8} {'Method':<12} {'CV Acc':>10}  {'Test Acc':>10}")
print(f"  {'-'*46}")
for cond, res in [('Pre', res_pre), ('Post', res_post)]:
    for method in ['MDM', 'TS + LDA']:
        m = res[method]
        print(f"  {cond:<8} {method:<12} "
              f"{m['cv_mean']*100:>9.1f}%  {m['test_acc']*100:>9.1f}%")
print()
for method in ['MDM', 'TS + LDA']:
    delta = res_post[method]['test_acc'] - res_pre[method]['test_acc']
    sign  = '+' if delta >= 0 else ''
    print(f"  {method:<12}  Δ test (post − pre) = {sign}{delta*100:.1f}%")
print('=' * 60)